# Nested feature-selection analysis

This read-only notebook audits the nested arm of the derived 8.3 feature-selection 2.0 rerun. Inner ranking uses 2017–2020 only, while 2021–2022 remains reserved for outer feature-count and stopping decisions.


## Locate nested artifacts

The canonical runner writes nested outputs under a separate artifact root and freezes its V0-full router fitted on inner training data. Until the full run completes, this notebook reports missing state without falling back to another experiment.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.3-feature-selection-2.0"
ARTIFACT_ROOT = EXP_DIR / "artifacts/nested"
print("Nested artifacts available:", ARTIFACT_ROOT.exists())

Nested artifacts available: True


## Compare inner and outer decisions

The outer table is the key leakage safeguard inherited from the 2.2 methodology. Candidate lists are generated entirely inside the training period and then scored on disjoint future years and held-out station groups.

In [2]:
sys.path.insert(0, str(EXP_DIR))
from generate_results import load_selection_summary

summary = load_selection_summary("nested")
display(pd.DataFrame(summary["datasets"]))
print("Source:", ARTIFACT_ROOT / "selection_summary.json")

,dataset,global_n_features,outer_stopping_reason,regimes,router
0,derived_8.0,40,minimum_outer_upper_confidence_bound,{},NaN
1,derived_8.3,125,minimum_outer_upper_confidence_bound,"{'0': {'n_features': 125, 'n_delta': 0, 'outer...","{'kind': 'clustering_v0_full_k2', 'feature_sou..."


Source: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.3-feature-selection-2.0/artifacts/nested/selection_summary.json


## Inspect locked-learner, crossed-fold, and MoE diagnostics

These retrospective diagnostics compare the locked final learner, independent forward-time and station-time paths, progressive elimination, and frozen versus refit V0-full routing. They do not seed or bypass the selector.

In [3]:
from generate_results import build_candidate_ceiling_table

ceiling = build_candidate_ceiling_table()
display(ceiling)
print("Sources: artifacts/*/candidate_diagnostics/global_candidates.csv")

,artifact_set,dataset,full_train_diagnostic_n_features,full_train_diagnostic_retrospective_R2,retrospective_ceiling_n_features,retrospective_ceiling_R2
0,crossed_candidates_locked_outer,derived_8.0,150,0.797157,40,0.800203
1,crossed_candidates_locked_outer,derived_8.3,80,0.586301,150,0.596011
2,nested,derived_8.0,65,0.770186,150,0.776693
3,nested,derived_8.3,100,0.542999,65,0.563457
4,progressive_crossed_locked_outer,derived_8.0,100,0.818642,80,0.820916


Sources: artifacts/*/candidate_diagnostics/global_candidates.csv


In [4]:
from generate_results import build_moe_table

moe = build_moe_table()
display(moe)
print("Source:", ARTIFACT_ROOT / "retrospective_test_eval/metrics_summary.csv")

,artifact_set,dataset,model,beta,R2,RMSE,ubRMSE,Bias,MAE,Med|Err|,Pearson
17,nested,derived_8.3,2.0_clustering_v0_full_k2_frozen_shared_only,0.0,0.555561,0.069279,0.067872,-0.013890,0.050073,0.036506,0.767890
18,nested,derived_8.3,2.0_clustering_v0_full_k2_refit_shared_plus_delta,0.0,0.555561,0.069279,0.067872,-0.013890,0.050073,0.036506,0.767890
16,nested,derived_8.3,2.0_clustering_v0_full_k2_shared_plus_delta,0.0,0.555561,0.069279,0.067872,-0.013890,0.050073,0.036506,0.767890
14,nested,derived_8.3,2.0_global,0.0,0.554386,0.069371,0.067910,-0.014162,0.050092,0.037195,0.767698


Source: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.3-feature-selection-2.0/artifacts/nested/retrospective_test_eval/metrics_summary.csv
